## 3. System Level Intergration Using PYNQ Flow

### 3.1 Load Overlay

The `Overlay` library encapsulates the interface for interaction between the ARM CPU and the FPGA's PL section.

We can load the generated hardware design onto the PL simply using the `Overlay()` function.

With the statement `overlay.lenet5_0`, we can interact with the IP in the form of accessing Python objects.

In [ ]:
from pynq import Overlay
from pynq import allocate
import numpy as np
import cv2
from PIL import Image as PIL_Image
from PIL import ImageEnhance
from PIL import ImageOps
from scipy import misc

In [ ]:
overlay = Overlay('lenet5.bit')
top = overlay.lenet5_0

In [ ]:
top.register_map

### 3.2 Generate binary image array

You can use the following script to translate a image from MNIST dataset to binary image array, the example image is as follow.

![Example MNIST Image](./image/mnist_image.png)

In [ ]:
from PIL import Image
import numpy as np

def image_to_binary_array(image_path, threshold=128):
    # Open and convert image to grayscale
    img = Image.open(image_path).convert('L')
    
    # Ensure image is 28x28
    if img.size != (28, 28):
        img = img.resize((28, 28))
    
    # Convert to numpy array
    pixel_array = np.array(img)
    
    # Create binary array (1 for white-ish pixels, 0 for dark pixels)
    binary_array = (pixel_array > threshold).astype(int)
    
    # Convert each row to a single integer (28 bits)
    result = []
    for row in binary_array:
        binary_num = 0
        for bit in row:
            binary_num = (binary_num << 1) | bit
        result.append(binary_num)
    
    return result

# Example usage
image_path = "./image/mnist_image.png"
binary_array = image_to_binary_array(image_path)
    
# Print results
print("Binary array (28 numbers, each representing a row):")
for i, num in enumerate(binary_array):
    print(f"0b{bin(num)[2:].zfill(28)}")

### 3.3 Process the image

In [ ]:
# input image
# Initialize the images array with the provided binary values

image = binary_array

input_array = np.zeros((28 * 28 * 1), dtype=int)

for y in range(28):
    v = image[y]
    for x in range(28):
        input_array[y * 28 + x] = (v >> (27 - x)) & 1
            
   

In [ ]:
# Define the path to the file
file_path1 = 'conv0_weight.txt'

# Read the file content
with open(file_path1, 'r') as file:
    lines = file.readlines()

# Extract the array content between the braces
array_content = ''.join(lines).split('{')[1].split('}')[0]

# Clean the content, filter out empty strings, and convert to integers
array_values = list(map(int, filter(None, array_content.replace('\n', '').replace(' ', '').split(','))))

# Convert to a NumPy array and reshape it to the desired shape (16, 5, 5, 1)
conv0_weight = np.array(array_values).reshape(16, 5, 5, 1)

# Print the NumPy array to verify the result
# print(array_values)

# Define the path to the file
file_path2 = 'conv1_weight.txt'

# Read the file content
with open(file_path2, 'r') as file:
    lines = file.readlines()

# Extract the array content between the braces
array_content = ''.join(lines).split('{')[1].split('}')[0]

# Clean the content, filter out empty strings, and convert to integers
array_values = list(map(int, filter(None, array_content.replace('\n', '').replace(' ', '').split(','))))

# Convert to a NumPy array and reshape it to the desired shape (16, 5, 5, 1)
conv1_weight = np.array(array_values).reshape(16, 5, 5, 16)

# Print the NumPy array to verify the result
# print(array_values)

# Define the path to the file
file_path3 = 'matmul0_weight.txt'

# Read the file content
with open(file_path3, 'r') as file:
    lines = file.readlines()

# Extract the array content between the braces
array_content = ''.join(lines).split('{')[1].split('}')[0]

# Clean the content, filter out empty strings, and convert to integers
array_values = list(map(int, filter(None, array_content.replace('\n', '').replace(' ', '').split(','))))

# Convert to a NumPy array and reshape it to the desired shape (16, 5, 5, 1)
matmul0_weight = np.array(array_values).reshape(10, 256)

# Print the NumPy array to verify the result
# print(matmul0_weight)

Allocate memory for IP

In [ ]:
# allocate memory

input_buffer = allocate(shape= (1* 28 *28), dtype='int32')
conv0_weight_buffer = allocate(shape= (16* 5* 5* 1), dtype='int32')
conv1_weight_buffer = allocate(shape= (16* 5* 5* 16), dtype='int32')
matmul0_weight_buffer = allocate(shape= (10* 256), dtype='int32')
output_buffer = allocate(shape= (1* 10), dtype='int32')

# copy data into memory
np.copyto(input_buffer, input_array.flatten())
np.copyto(conv0_weight_buffer, conv0_weight.flatten())
np.copyto(conv1_weight_buffer, conv1_weight.flatten())
np.copyto(matmul0_weight_buffer, matmul0_weight.flatten())

In [ ]:
import time

top.register_map.CTRL.AP_START = 1

start_time = time.time()

## copy input
dma_input = overlay.axi_dma_0
dma_conv0_weight = overlay.axi_dma_1
dma_conv1_weight = overlay.axi_dma_2
dma_fc_weight = overlay.axi_dma_3
dma_output = overlay.axi_dma_4

dma_input.sendchannel.transfer(input_buffer)
dma_conv0_weight.sendchannel.transfer(conv0_weight_buffer)
dma_conv1_weight.sendchannel.transfer(conv1_weight_buffer)
dma_fc_weight.sendchannel.transfer(matmul0_weight_buffer)
dma_output.recvchannel.transfer(output_buffer)

dma_input.sendchannel.wait() # wait for send channel
dma_conv0_weight.sendchannel.wait() # wait for send channel
dma_conv1_weight.sendchannel.wait() # wait for send channel
dma_fc_weight.sendchannel.wait() # wait for send channel
dma_output.recvchannel.wait() # wait for recv channel

end_time = time.time()

print("Time: {}s".format(end_time - start_time))

In [ ]:
# Print the output buffer for reference
print("Output buffer:", output_buffer)

# Finding the max value and its index in the 1D output_buffer
max_index = np.argmax(output_buffer)
max_value = output_buffer[max_index]

print(f"Output label = {max_index} ({max_value})")


---------------------------------------
<p align="center">Copyright&copy; 2024 Advanced Micro Devices</p>